In [4]:
import pyodbc
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

# Connection string
conn_str = (
    r'DRIVER={ODBC Driver 17 for SQL Server};'
    r'SERVER=QUAN;'
    r'DATABASE=gt;'
    r'Trusted_Connection=yes;'
)

# SQL query
query = """
SELECT 
    call_type, priority, initial_call_type_mapping,
    dispatch_precinct, dispatch_sector, dispatch_beat,
    dispatch_reporting_area, cad_event_response_category,
    call_type_indicator, dispatch_Neighborhood,
    call_type_received_classification,
    cad_event_original_time_queued_date,
    call_sign_total_service_time_s
FROM [dbo].[call_data_20251019_processed_v44]
TABLESAMPLE (10 PERCENT)
"""

# Fetch data
with pyodbc.connect(conn_str) as conn:
    df = pd.read_sql(query, conn)

# Clean data
df = df.dropna(subset=['call_sign_total_service_time_s'])

# Categorical encoding
categorical_cols = [
    'call_type', 'priority', 'initial_call_type_mapping',
    'dispatch_precinct', 'dispatch_sector', 'dispatch_beat',
    'dispatch_reporting_area', 'cad_event_response_category',
    'call_type_indicator', 'dispatch_Neighborhood',
    'call_type_received_classification'
]
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# Split data
X = df.drop('call_sign_total_service_time_s', axis=1)
y = df['call_sign_total_service_time_s']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Convert date to timestamp
X_train['cad_event_original_time_queued_date'] = pd.to_datetime(X_train['cad_event_original_time_queued_date']).astype('int64') / 10**9
X_test['cad_event_original_time_queued_date'] = pd.to_datetime(X_test['cad_event_original_time_queued_date']).astype('int64') / 10**9

# Fill NaNs
X_train.fillna(0, inplace=True)
X_test.fillna(0, inplace=True)

# Train model
params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.9
}

train_data = lgb.Dataset(X_train, label=y_train)
test_data = lgb.Dataset(X_test, label=y_test)

model = lgb.train(params, train_data, valid_sets=[test_data], num_boost_round=100)

# Evaluate model
y_pred = model.predict(X_test)

# Calculate metrics
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Model Performance Metrics:")
print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")
print(f"R²: {r2:.2f}")

C:\Users\RQ\AppData\Local\Temp\ipykernel_12124\163195421.py:32: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Model Performance Metrics:
RMSE: 2277.36
MAE: 1764.96
R²: 0.07
